# Speaker Analysis Models

Три задачі, один спільний feature extractor:

```
waveform (80000,)
    ↓  MelSpectrogram (hop=160 → 10ms)
features (80, 500)   ← 80 mel bins, 500 frames
    ↓  TDNN backbone
hidden (256, 500)
    ↓              ↓              ↓
 VAD head     Seg head      Pool + ArcFace
(500, 1)     (500, 4)       (256,) embedding
```

**FRAME_SHIFT = 0.01 (10 ms)** — узгоджено з hop_length=160 при SR=16000.

In [1]:
import torch
import warnings
warnings.filterwarnings("ignore", message="An output with one or more elements was resized")
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from itertools import permutations
import json

# ── константи (мають збігатись з preprocess_dataset.ipynb) ──
SR           = 16000
CHUNK_SEC    = 5.0
CHUNK_SAMPLES = int(CHUNK_SEC * SR)   # 80000
N_MELS       = 80
HOP_LENGTH   = 160    # 10 ms при SR=16000
WIN_LENGTH   = 400    # 25 ms
N_FFT        = 512
N_FRAMES     = int(CHUNK_SEC * SR / HOP_LENGTH)  # 500
MAX_SPEAKERS = 4
HIDDEN_DIM   = 256
EMBED_DIM    = 256
DATA_DIR     = Path("data")

print(f"Frames per chunk: {N_FRAMES}")  # має бути 500

Frames per chunk: 500


## 1. Dataset

In [2]:
class AMIChunkDataset(Dataset):
    """
    Читає .npz чанки з data/{split}/.
    Кожен чанк: audio (80000,), vad (500,), seg (500, 4).
    single_speaker_only=True — тільки чанки де активний рівно 1 спікер
    (використовується для навчання ембедінгів).
    """
    def __init__(self, split: str, min_speech_ratio: float = 0.1,
                 spk2idx: dict = None, single_speaker_only: bool = False):
        self.path = DATA_DIR / split
        with open(self.path / "meta.json") as f:
            meta = json.load(f)

        self.spk2idx = spk2idx or {}

        chunks = [c for c in meta["chunks"] if c["speech_ratio"] >= min_speech_ratio]

        if single_speaker_only:
            # лише чанки де в межах вікна був рівно 1 спікер
            chunks = [c for c in chunks if len(c["speakers"]) == 1]

        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        npz   = np.load(self.path / f"{chunk['chunk_id']:07d}.npz")

        audio = torch.from_numpy(npz["audio"])
        if audio.ndim == 2:
            audio = audio.mean(dim=-1)  # стерео → моно
        vad   = torch.from_numpy(npz["vad"])        # (500,)
        seg   = torch.from_numpy(npz["seg"])        # (500, 4)

        dom_spk = chunk["speakers"][int(seg.sum(0).argmax())] if chunk["speakers"] else "__silence__"
        spk_id  = self.spk2idx.get(dom_spk, 0)

        return audio, vad, seg, spk_id


def build_spk2idx(split="train", single_speaker_only=False):
    """Будує глобальний словник speaker → int index з meta.json."""    
    with open(DATA_DIR / split / "meta.json") as f:
        meta = json.load(f)
    chunks = meta["chunks"]
    if single_speaker_only:
        chunks = [c for c in chunks if len(c["speakers"]) == 1]
    speakers = sorted({s for c in chunks for s in c["speakers"]})
    return {s: i for i, s in enumerate(speakers)}


# загальний spk2idx для VAD і Seg
spk2idx = build_spk2idx("train")
print(f"Unique speakers in train: {len(spk2idx)}")

train_ds = AMIChunkDataset("train", spk2idx=spk2idx)
val_ds   = AMIChunkDataset("val",   spk2idx=spk2idx)
test_ds  = AMIChunkDataset("test",  spk2idx=spk2idx)
print(f"Chunks — train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

# окремий датасет для ембедінгів — лише чисті single-speaker чанки
# УВАГА: val/test в AMI мають повністю інших спікерів (0 перетинів з train).
# Closed-set classification на нових спікерах завжди дає acc≈0.
# Тому: ділимо TRAIN 60/20/20 → emb_train / emb_val / emb_test (ті самі спікери).
from torch.utils.data import random_split as _random_split

spk2idx_emb = build_spk2idx("train", single_speaker_only=True)
_all_emb = AMIChunkDataset("train", spk2idx=spk2idx_emb, single_speaker_only=True)
_n      = len(_all_emb)
_n_val  = _n // 5        # 20%
_n_test = _n // 5        # 20%
_n_train = _n - _n_val - _n_test  # 60%
emb_train_ds, emb_val_ds, emb_test_ds = _random_split(
    _all_emb, [_n_train, _n_val, _n_test],
    generator=torch.Generator().manual_seed(42)
)
print(f"Emb chunks — train: {len(emb_train_ds)}  val: {len(emb_val_ds)}  test: {len(emb_test_ds)}")
print(f"Unique speakers (single-spk): {len(spk2idx_emb)}")

audio, vad, seg, spk_id = train_ds[0]
print(f"Sample: audio={audio.shape} vad={vad.shape} seg={seg.shape} spk_id={spk_id}")


Unique speakers in train: 147
Chunks — train: 99650  val: 12798  test: 11001
Emb chunks — train: 24254  val: 8084  test: 8084
Unique speakers (single-spk): 147
Sample: audio=torch.Size([80000]) vad=torch.Size([500]) seg=torch.Size([500, 4]) spk_id=79


## 2. Feature Extractor + TDNN Backbone

In [3]:
class MelFrontend(nn.Module):
    """waveform (B, T) → mel (B, N_MELS, T_frames)"""
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR,
            n_fft=N_FFT,
            win_length=WIN_LENGTH,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
        )

    def forward(self, x):            # x: (B, 80000)
        mel = self.mel(x)[..., :N_FRAMES]  # обрізаємо до рівно 500
        mel = (mel + 1e-6).log()     # log-mel
        # нормалізація per-utterance
        mel = (mel - mel.mean(dim=-1, keepdim=True)) / (mel.std(dim=-1, keepdim=True) + 1e-6)
        return mel                   # (B, 80, 500)


class TDNNLayer(nn.Module):
    """1D dilated conv → BN → ReLU"""
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, out_ch, kernel,
                              dilation=dilation,
                              padding=dilation * (kernel - 1) // 2)
        self.bn   = nn.BatchNorm1d(out_ch)

    def forward(self, x):    # x: (B, C, T)
        return F.relu(self.bn(self.conv(x)))


class TDNNBackbone(nn.Module):
    """
    Стек TDNN шарів з різними dilation.
    Вхід:  (B, N_MELS, T)  → вихід: (B, HIDDEN_DIM, T)
    Зберігає часову роздільність — T фреймів залишається.
    """
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            TDNNLayer(N_MELS,      HIDDEN_DIM, kernel=3, dilation=1),
            TDNNLayer(HIDDEN_DIM,  HIDDEN_DIM, kernel=3, dilation=2),
            TDNNLayer(HIDDEN_DIM,  HIDDEN_DIM, kernel=3, dilation=4),
            TDNNLayer(HIDDEN_DIM,  HIDDEN_DIM, kernel=3, dilation=1),
            TDNNLayer(HIDDEN_DIM,  HIDDEN_DIM, kernel=1, dilation=1),
        )

    def forward(self, x):    # (B, N_MELS, T) → (B, HIDDEN_DIM, T)
        return self.layers(x)


# перевірка розмірностей
dummy  = torch.randn(2, CHUNK_SAMPLES)
mel    = MelFrontend()(dummy)
hidden = TDNNBackbone()(mel)
print(f"MelFrontend: {dummy.shape} → {mel.shape}")
print(f"TDNNBackbone: {mel.shape} → {hidden.shape}")

MelFrontend: torch.Size([2, 80000]) → torch.Size([2, 80, 500])
TDNNBackbone: torch.Size([2, 80, 500]) → torch.Size([2, 256, 500])


## 3. VAD Model

Для кожного фрейму передбачає 0 (тиша) або 1 (мовлення).  
Loss: **Binary Cross-Entropy** з pos_weight для балансу класів.

In [4]:
class VADModel(pl.LightningModule):
    def __init__(self, lr=1e-3, pos_weight=2.0):
        super().__init__()
        self.save_hyperparameters()
        self.frontend  = MelFrontend()
        self.backbone  = TDNNBackbone()
        self.head      = nn.Conv1d(HIDDEN_DIM, 1, kernel_size=1)
        self.pos_weight = pos_weight

    def forward(self, audio):
        x = self.frontend(audio)
        x = self.backbone(x)
        x = self.head(x).squeeze(1)
        return x  # logits

    def _step(self, batch, stage):
        audio, vad, seg, _ = batch
        logits = self(audio)                               # (B, 500)
        pw   = torch.tensor(self.pos_weight, device=self.device)
        loss = F.binary_cross_entropy_with_logits(logits, vad, pos_weight=pw)

        preds = (logits.sigmoid() > 0.5)
        target = vad.bool()

        tp = (preds &  target).sum().float()
        fp = (preds & ~target).sum().float()
        fn = (~preds & target).sum().float()
        f1 = tp / (tp + 0.5 * (fp + fn) + 1e-8)
        acc = (preds == target).float().mean()

        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log(f"{stage}_acc",  acc,  prog_bar=True, on_step=False, on_epoch=True)
        self.log(f"{stage}_f1",   f1,   prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, _):   return self._step(batch, "train")
    def validation_step(self, batch, _): return self._step(batch, "val")
    def test_step(self, batch, _):       return self._step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


# перевірка
vad_model = VADModel()
out = vad_model(torch.randn(2, CHUNK_SAMPLES))
print(f"VAD output: {out.shape}")  # (2, 500)


VAD output: torch.Size([2, 500])


## 4. Segmentation Model (PIT-CE)

Для кожного фрейму передбачає які з 4 спікерів активні.  
**PIT** (Permutation Invariant Training): перебираємо всі 4! = 24 перестановки спікерів,
беремо ту що дає мінімальний loss — це вирішує проблему довільного порядку спікерів у мітках.

In [5]:
# усі перестановки для 4 спікерів — обчислюємо один раз
ALL_PERMS = list(permutations(range(MAX_SPEAKERS)))  # 24 варіанти

def pit_bce_loss(logits, target):
    """
    logits : (B, T, N_spk)  — вихід моделі
    target : (B, T, N_spk)  — мітки з .npz
    Для кожного семплу в батчі незалежно шукаємо найкращу перестановку.
    """
    B, T, N = logits.shape
    # loss для кожної перестановки: (B, 24)
    perm_losses = torch.stack([
        F.binary_cross_entropy_with_logits(
            logits,
            target[:, :, list(perm)],
            reduction="none"
        ).mean(dim=(1, 2))           # (B,)
        for perm in ALL_PERMS
    ], dim=1)                        # (B, 24)

    best_loss = perm_losses.min(dim=1).values   # (B,)
    return best_loss.mean()


class SegmentationModel(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.frontend = MelFrontend()
        self.backbone = TDNNBackbone()
        self.head     = nn.Conv1d(HIDDEN_DIM, MAX_SPEAKERS, kernel_size=1)

    def forward(self, audio):                    # (B, 80000)
        x = self.frontend(audio)                 # (B, 80, 500)
        x = self.backbone(x)                    # (B, 256, 500)
        x = self.head(x).permute(0, 2, 1)       # (B, 500, 4)
        return x                                 # logits

    def _step(self, batch, stage):
        audio, vad, seg, _ = batch
        logits = self(audio)                     # (B, 500, 4)
        loss   = pit_bce_loss(logits, seg)
        # DER-proxy: frame-level accuracy після argmax перестановки
        preds  = (logits.sigmoid() > 0.5).float()
        acc    = (preds == seg).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log(f"{stage}_acc",  acc,  prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, _):   return self._step(batch, "train")
    def validation_step(self, batch, _): return self._step(batch, "val")
    def test_step(self, batch, _):       return self._step(batch, "test")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


# перевірка
seg_model = SegmentationModel()
out = seg_model(torch.randn(2, CHUNK_SAMPLES))
print(f"Seg output: {out.shape}")   # (2, 500, 4)

dummy_target = (torch.rand(2, 500, 4) > 0.5).float()
loss = pit_bce_loss(out, dummy_target)
print(f"PIT-BCE loss: {loss.item():.4f}")


Seg output: torch.Size([2, 500, 4])
PIT-BCE loss: 0.6999


## 5. Speaker Embedding Model (ArcFace)

Encoder стискає весь чанк у один вектор через **attentive statistics pooling** —
зважене mid + std по часовій осі.  
**ArcFace** (AAM-Softmax) додає кутовий margin до CE loss, що робить ембедінги більш розділеними.

In [6]:
class AttentiveStatsPool(nn.Module):
    """(B, C, T) → (B, 2*C) — зважені mean + std по часу."""
    def __init__(self, in_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(in_dim, 128, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(128, in_dim, kernel_size=1),
            nn.Softmax(dim=-1),
        )

    def forward(self, x):             # (B, C, T)
        w    = self.attention(x)      # (B, C, T) — ваги по часу
        mean = (w * x).sum(dim=-1)    # (B, C)
        std  = (w * (x - mean.unsqueeze(-1)) ** 2).sum(dim=-1).clamp(min=1e-6).sqrt()
        return torch.cat([mean, std], dim=1)  # (B, 2*C)


class ArcFaceHead(nn.Module):
    """AAM-Softmax: cosine similarity + additive angular margin."""
    def __init__(self, embed_dim, n_classes, scale=64.0, margin=0.1):
        super().__init__()
        self.scale  = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.randn(n_classes, embed_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels=None):
        # x: (B, embed_dim) — вже L2-нормалізований
        cos_theta = F.linear(x, F.normalize(self.weight))  # (B, n_classes)
        if labels is None:
            return self.scale * cos_theta                   # inference

        theta       = cos_theta.clamp(-1 + 1e-7, 1 - 1e-7).acos()
        target_cos  = (theta + self.margin).cos()
        one_hot     = torch.zeros_like(cos_theta).scatter_(1, labels.view(-1, 1), 1)
        logits      = self.scale * (one_hot * target_cos + (1 - one_hot) * cos_theta)
        return logits                                       # (B, n_classes)


class EmbeddingModel(pl.LightningModule):
    def __init__(self, n_speakers: int, lr=1e-3, embed_dim=EMBED_DIM):
        super().__init__()
        self.save_hyperparameters()
        self.frontend  = MelFrontend()
        self.backbone  = TDNNBackbone()
        self.pool      = AttentiveStatsPool(HIDDEN_DIM)
        self.proj      = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, embed_dim),
            nn.BatchNorm1d(embed_dim),
        )
        self.arcface   = ArcFaceHead(embed_dim, n_speakers)

    def encode(self, audio):                     # (B, 80000) → (B, embed_dim)
        x = self.frontend(audio)                 # (B, 80, 500)
        x = self.backbone(x)                    # (B, 256, 500)
        x = self.pool(x)                        # (B, 512)
        x = self.proj(x)                        # (B, embed_dim)
        return F.normalize(x, dim=-1)            # L2-нормалізація

    def forward(self, audio, labels=None):
        emb    = self.encode(audio)
        logits = self.arcface(emb, labels)
        return logits, emb

    def _step(self, batch, stage):
        audio, _, _, spk_id = batch
        spk_id = spk_id.long()
        # train: з ArcFace margin (навчання)
        # val/test: без margin → реальна accuracy на тих самих спікерах
        labels = spk_id if stage == "train" else None
        logits, _ = self(audio, labels)
        loss = F.cross_entropy(logits, spk_id)
        acc  = (logits.argmax(dim=-1) == spk_id).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log(f"{stage}_acc",  acc,  prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self._step(batch, "val")
        # Додатково: verification accuracy через cosine similarity всередині батча
        audio, _, _, spk_id = batch
        spk_id = spk_id.long()
        emb = self.encode(audio)                          # (B, D), L2-norm
        sim = torch.mm(emb, emb.t())                     # (B, B)
        B   = spk_id.shape[0]
        mask = ~torch.eye(B, dtype=torch.bool, device=self.device)
        same = (spk_id.unsqueeze(0) == spk_id.unsqueeze(1)) & mask
        diff = ~(spk_id.unsqueeze(0) == spk_id.unsqueeze(1)) & mask
        if same.any() and diff.any():
            ver_acc = torch.cat([
                (sim[same] > 0.0),   # same-speaker пари мають cos>0
                (sim[diff] < 0.0),   # diff-speaker пари мають cos<0
            ]).float().mean()
            self.log("val_ver_acc", ver_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, _):   return self._step(batch, "train")
    def test_step(self, batch, _):       return self._step(batch, "test")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10, eta_min=1e-6)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}


# перевірка
N_SPK = len(spk2idx)
emb_model = EmbeddingModel(n_speakers=N_SPK)
logits, emb = emb_model(torch.randn(2, CHUNK_SAMPLES),
                         torch.tensor([0, 1]))
print(f"Embedding: {emb.shape}")   # (2, 256)
print(f"Logits:    {logits.shape}") # (2, N_SPK)

Embedding: torch.Size([2, 256])
Logits:    torch.Size([2, 147])


## 6. DataLoaders

In [7]:
BATCH_SIZE = 32

# автоматичне визначення пристрою
if torch.cuda.is_available():
    DEVICE     = "cuda"
    NUM_WORKERS = 4
    PIN_MEMORY  = True
elif torch.backends.mps.is_available():
    DEVICE      = "mps"
    NUM_WORKERS = 0    # multiprocessing не працює в Jupyter на MPS
    PIN_MEMORY  = False
else:
    DEVICE      = "cpu"
    NUM_WORKERS = 0
    PIN_MEMORY  = False

print(f"Device: {DEVICE}  |  num_workers: {NUM_WORKERS}  |  pin_memory: {PIN_MEMORY}")

dl_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                 pin_memory=PIN_MEMORY,
                 persistent_workers=(NUM_WORKERS > 0))

train_loader = DataLoader(train_ds, shuffle=True,  **dl_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **dl_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **dl_kwargs)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Device: mps  |  num_workers: 0  |  pin_memory: False
Train batches: 3115
Val batches:   400
Test batches:  344


## 7. Тренування

In [9]:
# ── VAD ──────────────────────────────────────────────────
vad_model  = VADModel(lr=1e-3, pos_weight=2.0)
vad_logger = pl.loggers.CSVLogger(save_dir="models", name="vad")

vad_trainer = pl.Trainer(
    max_epochs=5,
    accelerator="auto",
    log_every_n_steps=10,
    logger=vad_logger,
    callbacks=[
        pl.callbacks.ModelCheckpoint(
            monitor="val_f1", mode="max", save_top_k=1,
            dirpath=f"{vad_logger.log_dir}/checkpoints",
            filename="epoch{epoch:02d}-f1={val_f1:.3f}",
        ),
        pl.callbacks.EarlyStopping(monitor="val_f1", mode="max", patience=3),
    ],
)
vad_trainer.fit(vad_model, train_loader, val_loader)
print(f"Logs + checkpoints: {vad_logger.log_dir}")


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ frontend │ MelFrontend  │      0 │ train │     0 │
│ 1 │ backbone │ TDNNBackbone │  720 K │ train │     0 │
│ 2 │ head     │ Conv1d       │    257 │ train │     0 │
└───┴──────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 720 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 720 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 22                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=5` reached.


Logs + checkpoints: models/vad/version_5


In [11]:
# ── Segmentation ─────────────────────────────────────────
seg_model  = SegmentationModel(lr=1e-3)
seg_logger = pl.loggers.CSVLogger(save_dir="models", name="seg")

seg_trainer = pl.Trainer(
    max_epochs=5,
    accelerator="auto",
    log_every_n_steps=10,
    logger=seg_logger,
    callbacks=[
        pl.callbacks.ModelCheckpoint(
            monitor="val_loss", mode="min", save_top_k=1,
            dirpath=f"{seg_logger.log_dir}/checkpoints",
            filename="epoch{epoch:02d}-loss={val_loss:.3f}",
        ),
        pl.callbacks.EarlyStopping(monitor="val_loss", patience=3),
    ],
)
seg_trainer.fit(seg_model, train_loader, val_loader)
print(f"Logs + checkpoints: {seg_logger.log_dir}")


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ frontend │ MelFrontend  │      0 │ train │     0 │
│ 1 │ backbone │ TDNNBackbone │  720 K │ train │     0 │
│ 2 │ head     │ Conv1d       │  1.0 K │ train │     0 │
└───┴──────────┴──────────────┴────────┴───────┴───────┘

Trainable params: 721 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 721 K                                                                                                
Total estimated model params size (MB): 2                                                                          
Modules in train mode: 22                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=5` reached.


Logs + checkpoints: models/seg/version_4


In [12]:
# ── Embeddings ───────────────────────────────────────────
emb_model  = EmbeddingModel(n_speakers=len(spk2idx_emb), lr=1e-4)
emb_logger = pl.loggers.CSVLogger(save_dir="models", name="emb")

emb_dl_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     pin_memory=PIN_MEMORY, persistent_workers=(NUM_WORKERS > 0))
emb_train_loader = DataLoader(emb_train_ds, shuffle=True,  **emb_dl_kwargs)
emb_val_loader   = DataLoader(emb_val_ds,   shuffle=False, **emb_dl_kwargs)

emb_trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    log_every_n_steps=10,
    logger=emb_logger,
    callbacks=[
        pl.callbacks.ModelCheckpoint(
            monitor="val_acc", mode="max", save_top_k=1,
            dirpath=f"{emb_logger.log_dir}/checkpoints",
            filename="epoch{epoch:02d}-acc={val_acc:.3f}",
        ),
        pl.callbacks.EarlyStopping(monitor="val_acc", mode="max", patience=3),
    ],
)
emb_trainer.fit(emb_model, emb_train_loader, emb_val_loader)
print(f"Logs + checkpoints: {emb_logger.log_dir}")


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ frontend │ MelFrontend        │      0 │ train │     0 │
│ 1 │ backbone │ TDNNBackbone       │  720 K │ train │     0 │
│ 2 │ pool     │ AttentiveStatsPool │ 65.9 K │ train │     0 │
│ 3 │ proj     │ Sequential         │  131 K │ train │     0 │
│ 4 │ arcface  │ ArcFaceHead        │ 37.6 K │ train │     0 │
└───┴──────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 956 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 956 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 31                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=10` reached.


Logs + checkpoints: models/emb/version_4


## 8. Оцінка на test set

In [14]:
emb_test_loader = DataLoader(emb_test_ds, shuffle=False, **emb_dl_kwargs)

In [15]:
print("VAD:")
# vad_model = VADModel.load_from_checkpoint("models/vad/version_0/checkpoints/epoch03-f1=0.847.ckpt")    
vad_trainer.test(vad_model, test_loader)

print("\nSegmentation:")
# seg_model = SegmentationModel.load_from_checkpoint("models/seg/version_0/checkpoints/...")  
seg_trainer.test(seg_model, test_loader)

print("\nEmbeddings:")
# emb_model = EmbeddingModel.load_from_checkpoint("models/emb/version_0/checkpoints/...", n_speakers=len(spk2idx))
emb_trainer.test(emb_model, emb_test_loader)

Output()

VAD:


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9139987826347351     │
│          test_f1          │    0.9475255608558655     │
│         test_loss         │    0.2461959272623062     │
└───────────────────────────┴───────────────────────────┘

Output()


Segmentation:


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8793819546699524     │
│         test_loss         │    0.1950952559709549     │
└───────────────────────────┴───────────────────────────┘

Output()


Embeddings:


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9658584594726562     │
│         test_loss         │    0.18951937556266785    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.18951937556266785, 'test_acc': 0.9658584594726562}]